# Baseline age model: train and evaluate on validation (tutorial)

This notebook is for **competitors and teachers** learning how the repository’s baseline pipeline works.

**What you will do**
1. Check that split pseudobulk files exist (`train` + `val`; optional `test`).
2. Build a temporary merged input file for `models/train_age_model.py`.
3. Train the baseline Random Forest.
4. Evaluate predictions against **validation** truth with `models/evaluate_val.py`.

**How to run**
- Open this notebook from the project root `aging-challenge-2026/`, or the first code cell sets paths relative to the repo.
- Execute cells **in order** (top to bottom).

**Time / resources**  
- Training uses CPU RAM and can take several minutes depending on `--n-genes` and `--n-estimators`. On a login node, use modest settings first (defaults are reasonable).

## 1. Concepts: what the baseline does

### Input data
- **Files:** split donor-aggregated pseudobulk under `data/scRNA-seq_pseudobulk/`:
  - `train_pseudobulk_donor_aggregated_public.h5ad`
  - `val_pseudobulk_donor_aggregated_public.h5ad`
- **Rows:** one row per **donor** (not per cell).
- **Columns (features):** for each gene, there are **five** pseudobulk expression values—one per **cell-type group** (e.g. CD4 T, CD8 T, NK, B, Monocytes). Feature names look like `ENSG...__CellTypeGroup`.
- `age` is intentionally stripped from these public pseudobulk h5ad files. Training/evaluation labels come from CSV files:
  - `data/metadata/train_age.csv`
  - `data/metadata/val_age.csv`
- This notebook injects ages from those CSVs and merges split files into one temporary `.h5ad` with `_split = train/val/test` so `train_age_model.py` works as intended.

### Training script (`train_age_model.py`)
- **Model:** scikit-learn `RandomForestRegressor` (regression → predicted age in years).
- **Feature selection (default):** keeps the top **N genes by variance** across donors (then keeps all five cell-type columns for those genes). This reduces dimensionality and runtime.
- **Alternative:** `--all-features` uses every gene (much heavier).
- **Preprocessing:** `log1p` on pseudobulk counts.
- **Splitting:** `train` rows fit the model; `val` rows report validation metrics.

### Evaluation script (`evaluate_val.py`)
- Merges predictions with **validation** ground-truth ages by `donor_id`.
- **Ground-truth source:** `data/metadata/val_age.csv` (`donor_id`, `age`).
- Reports **MAE**, **RMSE**, **R²**, **Pearson**, **Spearman** and writes `val_eval_metrics.csv` next to the predictions file.

## 2. Environment: paths and Python

We locate the **repository root** so subprocess calls match `python models/train_age_model.py` from the project root.

**Teaching note:** In HPC environments, activate the same conda/env that has `scanpy`, `sklearn`, `scipy` installed before starting Jupyter.

In [ ]:
import sys
import subprocess
from pathlib import Path

# If the notebook lives in notebooks/, parents[1] is the repo root
NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find models/train_age_model.py. Open Jupyter from the repo root or from notebooks/."
    )

DATA_DIR = PROJ_ROOT / "data"
RESULTS_DIR = PROJ_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJ_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Python:", sys.executable)

## 3. Check prerequisites

Before training, you need split donor-aggregated pseudobulk files:
- `data/scRNA-seq_pseudobulk/train_pseudobulk_donor_aggregated_public.h5ad`
- `data/scRNA-seq_pseudobulk/val_pseudobulk_donor_aggregated_public.h5ad`

For labels, you need:
- `data/metadata/train_age.csv` (columns: `donor_id,age`)
- `data/metadata/val_age.csv` (columns: `donor_id,age`)

In [ ]:
import scanpy as sc
import pandas as pd

PB_DIR = DATA_DIR / "scRNA-seq_pseudobulk"
META_DIR = DATA_DIR / "metadata"
TRAIN_H5AD = PB_DIR / "train_pseudobulk_donor_aggregated_public.h5ad"
VAL_H5AD   = PB_DIR / "val_pseudobulk_donor_aggregated_public.h5ad"
TRAIN_AGE_CSV = META_DIR / "train_age.csv"
VAL_AGE_CSV   = META_DIR / "val_age.csv"
MERGED_H5AD = RESULTS_DIR / "merged_train_val_test_for_baseline.ipynb.h5ad"

print("Train pseudobulk:", TRAIN_H5AD, "exists=", TRAIN_H5AD.exists())
print("Val pseudobulk  :", VAL_H5AD,   "exists=", VAL_H5AD.exists())
print("Train age CSV   :", TRAIN_AGE_CSV, "exists=", TRAIN_AGE_CSV.exists())
print("Val age CSV     :", VAL_AGE_CSV,   "exists=", VAL_AGE_CSV.exists())

if not (TRAIN_H5AD.exists() and VAL_H5AD.exists()):
    raise FileNotFoundError(
        "Need both train and val donor-aggregated h5ad files under data/scRNA-seq_pseudobulk/."
    )
if not (TRAIN_AGE_CSV.exists() and VAL_AGE_CSV.exists()):
    raise FileNotFoundError("Need train_age.csv and val_age.csv under data/metadata/.")

train_age = pd.read_csv(TRAIN_AGE_CSV)
val_age = pd.read_csv(VAL_AGE_CSV)
for df, name in [(train_age, 'train_age.csv'), (val_age, 'val_age.csv')]:
    if not {'donor_id', 'age'}.issubset(df.columns):
        raise ValueError(f"{name} must contain columns donor_id, age")
    df['donor_id'] = df['donor_id'].astype(str)
    df['age'] = pd.to_numeric(df['age'], errors='coerce')

# Build one merged file expected by train_age_model.py (needs _split train/val/test in one AnnData)
ad_tr = sc.read_h5ad(TRAIN_H5AD)
ad_va = sc.read_h5ad(VAL_H5AD)

ad_tr.obs['donor_id'] = ad_tr.obs['donor_id'].astype(str)
ad_va.obs['donor_id'] = ad_va.obs['donor_id'].astype(str)

tr_map = train_age.set_index('donor_id')['age']
va_map = val_age.set_index('donor_id')['age']
ad_tr.obs['age'] = ad_tr.obs['donor_id'].map(tr_map)
ad_va.obs['age'] = ad_va.obs['donor_id'].map(va_map)

if ad_tr.obs['age'].isna().any():
    missing = ad_tr.obs.loc[ad_tr.obs['age'].isna(), 'donor_id'].head(5).tolist()
    raise ValueError(f"Missing train ages for donor_id(s), e.g. {missing}")
if ad_va.obs['age'].isna().any():
    missing = ad_va.obs.loc[ad_va.obs['age'].isna(), 'donor_id'].head(5).tolist()
    raise ValueError(f"Missing val ages for donor_id(s), e.g. {missing}")

ad_tr.obs['_split'] = 'train'
ad_va.obs['_split'] = 'val'

# For this notebook we always evaluate on VAL, so we build a pseudo-test from val donors
# (train_age_model.py writes predictions for _split=='test').
parts = [ad_tr, ad_va]
ad_te = ad_va.copy()
ad_te.obs['_split'] = 'test'
ad_te.obs['age'] = float('nan')
parts.append(ad_te)
print("Using val donors as pseudo-test so test_predictions.csv can be scored against val labels.")

ad_merged = sc.concat(parts, axis=0, join='inner')
ad_merged.write_h5ad(MERGED_H5AD)

H5AD_PATH = MERGED_H5AD
TRUTH_PATH = VAL_AGE_CSV

print("\nMerged training input:", H5AD_PATH)
print("Merged shape:", ad_merged.n_obs, "rows ×", ad_merged.n_vars, "features")
print("Split counts:")
print(ad_merged.obs['_split'].value_counts().to_string())
print("\nNote: notebook output cells may show old paths from previous runs; trust the paths printed above.")

## 4. Run training (`train_age_model.py`)

This cell runs the same entry point as the command line, but uses the merged file built above (`train` + `val` + optional `test`/pseudo-test):

```bash
python models/train_age_model.py --input <merged_train_val_test.h5ad> [options]
```

**Parameters you can change below**
- `N_GENES` — how many highest-variance genes to keep (ignored if `ALL_FEATURES=True`).
- `N_ESTIMATORS` — number of trees in the forest (more trees → often stabler but slower).
- `ALL_FEATURES` — set `True` to disable gene filtering (large memory).
- `OUTPUT_DIR` — set to `None` for a timestamped folder under `results/`; or a fixed path string for reproducible reruns.

Validation metrics are printed by the training script from the `val` split.

In [ ]:
from datetime import datetime

# ---- Tune these for class demos / quick tests ----
N_GENES = 2000
N_ESTIMATORS = 100
ALL_FEATURES = False
SEED = 42
# Optional: set to a string path to pin outputs, e.g. str(RESULTS_DIR / "demo_run")
OUTPUT_DIR = None  # None → timestamped folder under results/

train_py = PROJ_ROOT / "models" / "train_age_model.py"
cmd = [sys.executable, str(train_py)]
cmd += ["--input", str(H5AD_PATH)]
cmd += ["--n-genes", str(N_GENES), "--n-estimators", str(N_ESTIMATORS), "--seed", str(SEED)]
if ALL_FEATURES:
    cmd.append("--all-features")
if OUTPUT_DIR:
    cmd += ["--output-dir", str(OUTPUT_DIR)]
else:
    run_dir = RESULTS_DIR / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    cmd += ["--output-dir", str(run_dir)]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJ_ROOT))
if result.returncode != 0:
    raise RuntimeError("train_age_model.py exited with code %s" % result.returncode)
print("\nTraining finished successfully.")

## 5. Locate the latest run and preview predictions

By default, this notebook writes outputs to `results/YYYY-MM-DD_HH-MM-SS/` (via `--output-dir`).

For this validation-focused workflow, we evaluate against val truth using predictions in `test_predictions.csv` from the merged input run.

**Files written**
- `test_predictions.csv` — `donor_id`, `predicted_age` for the pseudo-test block (val donors duplicated as `_split='test'` in the merged input)
- `rf_age_model.joblib` — fitted model
- `top_genes.csv` — top genes by Random Forest **feature importance** (aggregated across cell-type columns)

In [ ]:
import pandas as pd
from IPython.display import display

out_base = RESULTS_DIR
subdirs = sorted([d for d in out_base.iterdir() if d.is_dir()], reverse=True)
run_dir = None
pred_path = None
for d in subdirs:
    p = d / "test_predictions.csv"
    if p.exists():
        run_dir = d
        pred_path = p
        break

if pred_path is None:
    raise FileNotFoundError("No test_predictions.csv found under results/")

print("Run directory:", run_dir)
print("Predictions file:", pred_path)
display(pd.read_csv(pred_path).head(10))

## 6. Run evaluation (`evaluate_val.py`)

This merges predictions with validation labels from `data/metadata/val_age.csv` and prints regression metrics.

**Teaching notes**
- This notebook does **not** require test labels.
- For `--plot`, matplotlib must be installed; the PNG is saved beside the predictions.
- You can pass an explicit `--predictions` path to score a specific run folder.

In [ ]:
eval_py = PROJ_ROOT / "models" / "evaluate_val.py"

cmd = [
    sys.executable,
    str(eval_py),
    "--predictions", str(pred_path),
    "--truth-csv", str(TRUTH_PATH),
]
# Uncomment to save scatter plot (requires matplotlib):
# cmd.append("--plot")
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJ_ROOT))
if result.returncode != 0:
    raise RuntimeError("evaluate_val.py exited with code %s" % result.returncode)

## 7. Load saved metrics

Metrics are stored as `val_eval_metrics.csv` next to the predictions file. **Interpret briefly:**
- **MAE** — average error in years.
- **R²** — fraction of age variance explained (higher is better, capped at 1 for perfect predictions).
- **Pearson / Spearman** — correlation between predicted and true age linearly vs. by rank.

In [ ]:
from IPython.display import display

metrics_file = pred_path.parent / "val_eval_metrics.csv"
if metrics_file.exists():
    display(pd.read_csv(metrics_file))
else:
    print("No val_eval_metrics.csv yet (run evaluation cell).")

## 8. Optional: top genes from the last training run

Feature importances are **not** causal; they indicate how much the forest used each gene’s pseudobulk signal in this pipeline. High overlap with biological ageing literature can be a sanity check.

In [ ]:
from IPython.display import display

top_path = pred_path.parent / "top_genes.csv"
if top_path.exists():
    display(pd.read_csv(top_path))
else:
    print("No top_genes.csv beside this run.")

## Further ideas for competitors

- Swap the Random Forest for another regressor (e.g. gradient boosting, elastic net on PCA).
- Engineer features (pathways, cell-type proportions) instead of raw pseudobulk.
- Use cell-level models + pooling; compare to this donor-aggregated baseline.
- Keep test labels hidden; tune only on train/validation labels.

See `models/README.md` and the main `README.md` for paths and data structure.